# GRU soft sensor v14

生センサ / latentと、MLP / GRUを組み合わせた4モデルを比べる。

| 入力 | 推定器 | モデル名 |
|---|---|---|
| 生センサ＋操作入力 | MLP | `direct_mlp` |
| 生センサ＋操作入力 | GRU | `direct_gru` |
| latent＋操作入力 | MLP | `latent_mlp` |
| latent＋操作入力 | GRU | `latent_gru` |

v13ではLatent GRUがDirect GRUより良かった。ただ、GRU同士しか比べていないので、latentが効いたのか、推定器との組み合わせが良かったのかはまだ分からない。今回はヘッドにMLPも追加してそれを確かめる。

主な条件はAR型の出力外乱、latent 8次元、品質ラベル50点、5 seedにした。真値MAEは学習に使わず、シミュレーターの確認のためだけに見る。

## 1. Import

ライブラリです。

In [ ]:
import copy
import json
import math
import os
import platform
import random
import sys
import time
import traceback
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import mean_absolute_error
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 2. 設定

In [ ]:
@dataclass
class Config:
    run_mode: str = "smoke"  # smoke / full
    seeds: tuple = (42, 43, 44, 45, 46)

    T: int = 50_000
    train_ratio: float = 0.60
    val_ratio: float = 0.20

    quality_interval: int = 150
    quality_delay: int = 40
    encoder_length: int = 64
    quality_length: int = 64
    train_label_count: int = 50

    x_dim: int = 4
    u_dim: int = 2
    z_dim: int = 8
    hidden_dim: int = 48

    dynamics_rollout: int = 20
    dynamics_batch_size: int = 128
    quality_batch_size: int = 32

    dynamics_epochs: int = 35
    quality_epochs: int = 50
    curriculum_epochs: int = 15
    dynamics_patience: int = 8
    quality_patience: int = 10
    min_delta: float = 1e-5

    learning_rate: float = 1e-3
    weight_decay: float = 1e-5

    drift_amp: float = 0.12
    drift_rho: float = 0.995
    quality_noise_std: float = 0.75
    outlier_rate: float = 0.05
    outlier_std: float = 2.0

    show_progress: bool = True


CFG = Config()
if CFG.run_mode not in {"smoke", "full"}:
    raise ValueError("run_mode must be 'smoke' or 'full'")

if CFG.run_mode == "smoke":
    CFG.seeds = (42,)
    CFG.T = 1_200
    CFG.quality_interval = 30
    CFG.quality_delay = 8
    CFG.encoder_length = 12
    CFG.quality_length = 12
    CFG.train_label_count = 12
    CFG.hidden_dim = 12
    CFG.dynamics_rollout = 3
    CFG.dynamics_epochs = 2
    CFG.quality_epochs = 2
    CFG.curriculum_epochs = 2
    CFG.dynamics_patience = 2
    CFG.quality_patience = 2
    CFG.quality_batch_size = 8
    if not torch.cuda.is_available():
        torch.set_num_threads(1)

print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))

## 3. 保存と共通関数

結果の保存、seed固定、標準化、MAE計算をここにまとめる。

JSONとCSVは一時ファイルを挟んでから置き換える。途中で失敗しても、壊れた結果を残しにくくするため。実行環境もJSONへ記録する。

In [ ]:
RESULT_ROOT = Path("GRU_soft_sensor_simulation_v14_results")
RUN_DIR = RESULT_ROOT / CFG.run_mode
RUN_DIR.mkdir(parents=True, exist_ok=True)

(RUN_DIR / "RUNNING").write_text("running\n", encoding="utf-8")
for marker in ("COMPLETE", "FAILED"):
    path = RUN_DIR / marker
    if path.exists():
        path.unlink()


def save_json(name, obj):
    path = RUN_DIR / name
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    tmp.replace(path)


def save_csv(name, frame):
    path = RUN_DIR / name
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    tmp.replace(path)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def progress(iterable, desc, total=None, leave=False):
    return tqdm(
        iterable,
        desc=desc,
        total=total,
        leave=leave,
        dynamic_ncols=True,
        disable=not CFG.show_progress,
    )


def causal_moving_average(a, width):
    a = np.asarray(a, dtype=np.float32)
    c = np.cumsum(np.insert(a, 0, 0.0))
    out = np.empty_like(a)
    out[: width - 1] = c[1:width] / np.arange(1, width)
    out[width - 1 :] = (c[width:] - c[:-width]) / width
    return out.astype(np.float32)


def standardize_train(array, train_end, eps=1e-6):
    mean = array[:train_end].mean(axis=0, keepdims=True)
    std = array[:train_end].std(axis=0, keepdims=True)
    std = np.maximum(std, eps)
    scaled = (array - mean) / std
    return scaled.astype(np.float32), mean.astype(np.float32), std.astype(np.float32)


def safe_mae(y_true, y_pred):
    return float(mean_absolute_error(np.asarray(y_true), np.asarray(y_pred)))


def parameter_count(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


ENVIRONMENT = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "device": str(DEVICE),
}
save_json("config.json", asdict(CFG))
save_json("environment.json", ENVIRONMENT)

## 4. シミュレーター

仮想センサ4個と操作入力2個を作る。品質は内部状態の過去値から作り、AR外乱は品質出力だけに加える。

品質推定器にはセンサと操作入力だけを渡す。真の品質と外乱は入力に含めず、実データでは見えない値を使わないようにする。

In [ ]:
def make_ar_drift(T, seed, amp, rho):
    rng = np.random.default_rng(seed)
    innovation_std = amp * np.sqrt(max(1e-8, 1.0 - rho**2))
    drift = np.zeros(T, dtype=np.float32)
    drift[0] = rng.normal(0.0, amp)
    for t in range(1, T):
        drift[t] = rho * drift[t - 1] + rng.normal(0.0, innovation_std)
    return drift


def simulate_old_process(cfg, seed):
    process_rng = np.random.default_rng(seed)
    sensor_rng = np.random.default_rng(seed + 10_000)
    drift = make_ar_drift(
        cfg.T,
        seed + 20_000,
        cfg.drift_amp,
        cfg.drift_rho,
    )

    U = np.zeros((cfg.T, cfg.u_dim), dtype=np.float32)
    S = np.zeros((cfg.T, cfg.x_dim), dtype=np.float32)
    S[0] = np.array([55.0, 1.05, 1.00, 1.00], dtype=np.float32)

    sp_temp = 58.0 + 3.0 * np.sin(np.linspace(0, 5.5 * np.pi, cfg.T))
    sp_conc = 1.10 + 0.08 * np.sin(
        np.linspace(0, 4.0 * np.pi, cfg.T) + 1.0
    )

    for t in range(1, cfg.T):
        if t % 400 == 0:
            U[t] = U[t - 1] + process_rng.normal(0, [0.8, 0.4])
        else:
            U[t] = 0.995 * U[t - 1] + process_rng.normal(0, [0.02, 0.01])
        U[t] = np.clip(U[t], [-3.0, -2.0], [3.0, 2.0])

        temp, conc, press, visc = S[t - 1]
        pulse = process_rng.normal(0, 3.0) if process_rng.random() < 0.004 else 0.0

        temp_target = sp_temp[t] + 0.9 * U[t, 0] - 1.5 * U[t, 1]
        conc_target = sp_conc[t] + 0.06 * U[t, 1] - 0.02 * U[t, 0]
        press_target = 1.00 + 0.10 * U[t, 1] + 0.015 * (temp - 58.0)
        visc_target = 1.00 + 0.55 * (conc - 1.10) - 0.012 * (temp - 58.0)

        S[t, 0] = (
            temp
            + 0.035 * (temp_target - temp)
            + process_rng.normal(0, 0.20)
            + pulse
        )
        S[t, 1] = (
            conc
            + 0.045 * (conc_target - conc)
            + process_rng.normal(0, 0.008)
        )
        S[t, 2] = (
            press
            + 0.060 * (press_target - press)
            + process_rng.normal(0, 0.006)
        )
        S[t, 3] = (
            visc
            + 0.040 * (visc_target - visc)
            + process_rng.normal(0, 0.010)
        )

    X = S + sensor_rng.normal(
        0,
        [0.08, 0.003, 0.003, 0.004],
        size=S.shape,
    )

    signal = (
        0.22 * (causal_moving_average(S[:, 0], 45) - 58.0)
        + 4.0 * (causal_moving_average(S[:, 1], 35) - 1.10)
        - 2.5 * (causal_moving_average(S[:, 2], 25) - 1.00)
        + 1.8 * (causal_moving_average(S[:, 3], 60) - 1.00)
    )
    q_true = (93.0 + signal + drift).astype(np.float32)

    return X.astype(np.float32), U.astype(np.float32), q_true, drift, S


def build_data(cfg, seed):
    train_end = int(cfg.T * cfg.train_ratio)
    val_end = int(cfg.T * (cfg.train_ratio + cfg.val_ratio))

    X_raw, U_raw, q_true, drift, state_true = simulate_old_process(cfg, seed)
    X, X_mean, X_std = standardize_train(X_raw, train_end)
    U, U_mean, U_std = standardize_train(U_raw, train_end)

    start = max(60, cfg.encoder_length + cfg.quality_length)
    feature_times = np.arange(
        start,
        cfg.T - cfg.quality_delay,
        cfg.quality_interval,
        dtype=int,
    )
    measurement_times = feature_times + cfg.quality_delay

    measurement_rng = np.random.default_rng(seed + 30_000)
    q_obs = q_true[feature_times].copy()
    q_obs += measurement_rng.normal(
        0,
        cfg.quality_noise_std,
        size=len(q_obs),
    )
    outlier_mask = measurement_rng.random(len(q_obs)) < cfg.outlier_rate
    q_obs[outlier_mask] += measurement_rng.normal(
        0,
        cfg.outlier_std,
        size=int(outlier_mask.sum()),
    )

    base_train_mask = measurement_times < train_end
    q_val_mask = (feature_times >= train_end) & (measurement_times < val_end)
    q_test_mask = feature_times >= val_end

    train_indices = np.where(base_train_mask)[0]
    keep_count = min(cfg.train_label_count, len(train_indices))
    subset_rng = np.random.default_rng(seed + 40_000)
    selected = np.sort(subset_rng.permutation(train_indices)[:keep_count])
    q_train_mask = np.zeros_like(base_train_mask, dtype=bool)
    q_train_mask[selected] = True

    if not q_train_mask.any() or not q_val_mask.any() or not q_test_mask.any():
        raise ValueError("quality split is empty")

    q_mean = np.float32(q_obs[q_train_mask].mean())
    q_std = np.float32(max(q_obs[q_train_mask].std(), 1e-6))
    q_obs_scaled = ((q_obs - q_mean) / q_std).astype(np.float32)

    assert feature_times.min() >= cfg.encoder_length + cfg.quality_length
    assert np.all(measurement_times[q_train_mask] < train_end)
    assert np.all(feature_times[q_val_mask] >= train_end)
    assert np.all(measurement_times[q_val_mask] < val_end)
    assert np.all(feature_times[q_test_mask] >= val_end)

    return {
        "X": X,
        "U": U,
        "X_raw": X_raw,
        "U_raw": U_raw,
        "q_true": q_true,
        "q_obs": q_obs.astype(np.float32),
        "q_obs_scaled": q_obs_scaled,
        "q_mean": q_mean,
        "q_std": q_std,
        "drift": drift,
        "state_true": state_true,
        "feature_times": feature_times,
        "measurement_times": measurement_times,
        "q_train_mask": q_train_mask,
        "q_val_mask": q_val_mask,
        "q_test_mask": q_test_mask,
        "train_end": train_end,
        "val_end": val_end,
        "X_mean": X_mean,
        "X_std": X_std,
        "U_mean": U_mean,
        "U_std": U_std,
        "n_train_labels": int(q_train_mask.sum()),
        "n_val_labels": int(q_val_mask.sum()),
        "n_test_labels": int(q_test_mask.sum()),
    }

## 5. Dataset

センサ予測用と品質推定用でDatasetを分ける。

センサ予測では、過去履歴から将来センサを複数step予測する。品質推定では品質の対象時刻を `t` として、入力区間を `[t - L, t)` に固定する。時刻 `t` のセンサは入れず、未来情報が混ざらないようにする。

In [ ]:
class DynamicsDataset(Dataset):
    def __init__(self, X, U, start, end, history, rollout):
        self.X = torch.from_numpy(X)
        self.U = torch.from_numpy(U)
        self.history = history
        self.rollout = rollout
        lo = max(start, history)
        hi = min(end - rollout + 1, len(X) - rollout + 1)
        self.times = np.arange(lo, max(lo, hi), dtype=int)

    def __len__(self):
        return len(self.times)

    def __getitem__(self, index):
        t = int(self.times[index])
        return (
            self.X[t - self.history : t],
            self.U[t - self.history : t],
            self.U[t : t + self.rollout],
            self.X[t : t + self.rollout],
        )


class QualityDataset(Dataset):
    def __init__(self, features, targets, feature_times, history, mask):
        self.features = torch.from_numpy(features)
        self.targets = torch.from_numpy(targets.astype(np.float32))
        selected = np.where(mask)[0]
        self.times = feature_times[selected]
        self.label_indices = selected
        self.history = history

    def __len__(self):
        return len(self.times)

    def __getitem__(self, index):
        t = int(self.times[index])
        label_index = int(self.label_indices[index])
        return (
            self.features[t - self.history : t],
            self.targets[label_index],
            label_index,
        )

## 6. センサ予測モデル

センサと操作入力の履歴をGRU encoderへ入れ、8次元latentを作る。操作入力でlatentを進め、将来センサを最大20 step予測する。

学習の最初は短い予測から始め、少しずつrolloutを延ばす。いきなり20 stepを学習するより安定しやすそう。

ここでは品質ラベルを使わない。validationでは、モデル予測と直前値を維持するだけの予測を比べる。

In [ ]:
class ResidualLatentDynamics(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = nn.GRU(
            cfg.x_dim + cfg.u_dim,
            cfg.hidden_dim,
            batch_first=True,
        )
        self.to_z = nn.Linear(cfg.hidden_dim, cfg.z_dim)
        self.transition = nn.Sequential(
            nn.Linear(cfg.z_dim + cfg.u_dim, cfg.hidden_dim),
            nn.Tanh(),
            nn.Linear(cfg.hidden_dim, cfg.z_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(cfg.z_dim, cfg.hidden_dim),
            nn.ReLU(),
            nn.Linear(cfg.hidden_dim, cfg.x_dim),
        )

    def encode(self, x_history, u_history):
        sequence = torch.cat([x_history, u_history], dim=-1)
        _, hidden = self.encoder(sequence)
        return self.to_z(hidden[-1])

    def step(self, z, u):
        delta = self.transition(torch.cat([z, u], dim=-1))
        return z + delta

    def rollout(self, x_history, u_history, u_future, steps=None):
        z = self.encode(x_history, u_history)
        steps = u_future.shape[1] if steps is None else steps
        predictions = []
        for k in range(steps):
            z = self.step(z, u_future[:, k])
            predictions.append(self.decoder(z))
        return torch.stack(predictions, dim=1)


@torch.no_grad()
def evaluate_dynamics(model, data, cfg):
    dataset = DynamicsDataset(
        data["X"],
        data["U"],
        data["train_end"],
        data["val_end"],
        cfg.encoder_length,
        cfg.dynamics_rollout,
    )
    loader = DataLoader(dataset, batch_size=cfg.dynamics_batch_size)
    loss_fn = nn.MSELoss()

    model_losses = []
    persistence_losses = []
    model.eval()
    for x_history, u_history, u_future, target in loader:
        x_history = x_history.to(DEVICE)
        u_history = u_history.to(DEVICE)
        u_future = u_future.to(DEVICE)
        target = target.to(DEVICE)

        prediction = model.rollout(x_history, u_history, u_future)
        persistence = x_history[:, -1:, :].expand_as(target)

        model_losses.append(loss_fn(prediction, target).item())
        persistence_losses.append(loss_fn(persistence, target).item())

        if cfg.run_mode == "smoke":
            break

    return float(np.mean(model_losses)), float(np.mean(persistence_losses))


def train_dynamics(data, cfg, seed):
    set_seed(seed)
    model = ResidualLatentDynamics(cfg).to(DEVICE)

    train_dataset = DynamicsDataset(
        data["X"],
        data["U"],
        0,
        data["train_end"],
        cfg.encoder_length,
        cfg.dynamics_rollout,
    )
    val_dataset = DynamicsDataset(
        data["X"],
        data["U"],
        data["train_end"],
        data["val_end"],
        cfg.encoder_length,
        cfg.dynamics_rollout,
    )
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        raise ValueError("dynamics dataset is empty")

    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.dynamics_batch_size,
        shuffle=True,
        generator=generator,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.dynamics_batch_size,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    loss_fn = nn.MSELoss()

    best_value = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    wait = 0

    epochs = progress(
        range(cfg.dynamics_epochs),
        desc=f"dynamics seed={seed}",
        total=cfg.dynamics_epochs,
    )
    for epoch in epochs:
        if cfg.curriculum_epochs <= 1:
            rollout = cfg.dynamics_rollout
        else:
            fraction = min(1.0, epoch / float(cfg.curriculum_epochs - 1))
            rollout = 1 + int(round(fraction * (cfg.dynamics_rollout - 1)))

        model.train()
        for x_history, u_history, u_future, target in train_loader:
            x_history = x_history.to(DEVICE)
            u_history = u_history.to(DEVICE)
            u_future = u_future[:, :rollout].to(DEVICE)
            target = target[:, :rollout].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            prediction = model.rollout(
                x_history,
                u_history,
                u_future,
                steps=rollout,
            )
            loss = loss_fn(prediction, target)
            loss.backward()
            optimizer.step()

            if cfg.run_mode == "smoke":
                break

        model.eval()
        values = []
        with torch.no_grad():
            for x_history, u_history, u_future, target in val_loader:
                prediction = model.rollout(
                    x_history.to(DEVICE),
                    u_history.to(DEVICE),
                    u_future.to(DEVICE),
                )
                values.append(loss_fn(prediction, target.to(DEVICE)).item())
                if cfg.run_mode == "smoke":
                    break
        value = float(np.mean(values))

        curriculum_done = epoch >= cfg.curriculum_epochs - 1
        if curriculum_done and value < best_value - cfg.min_delta:
            best_value = value
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        elif curriculum_done:
            wait += 1

        epochs.set_postfix(
            rollout=rollout,
            val=f"{value:.4f}",
            best=(f"{best_value:.4f}" if np.isfinite(best_value) else "inf"),
        )
        if curriculum_done and wait >= cfg.dynamics_patience:
            break

    epochs.close()
    model.load_state_dict(best_state)
    model.eval()

    val_mse, persistence_mse = evaluate_dynamics(model, data, cfg)
    return model, val_mse, persistence_mse, best_epoch

## 7. Latent抽出と入力整合

学習済みDynamicsから全時刻のlatentを取り出し、train区間の統計量だけで標準化する。

`Z[t]` は `X[t-1]` までを見て作られるため、品質入力では `Z[s+1]` と `X[s]` を対応させる。これで1 stepのずれを避ける。

Direct入力には `[X, U]`、Latent入力には `[Z, U]` を使う。

In [ ]:
@torch.no_grad()
def compute_latents(model, X, U, cfg):
    model.eval()
    Z = np.full((len(X), cfg.z_dim), np.nan, dtype=np.float32)
    times = np.arange(cfg.encoder_length, len(X), dtype=int)

    batches = progress(
        range(0, len(times), 256),
        desc="latent extraction",
        total=math.ceil(len(times) / 256),
    )
    for start in batches:
        selected = times[start : start + 256]
        x_history = np.stack(
            [X[t - cfg.encoder_length : t] for t in selected]
        )
        u_history = np.stack(
            [U[t - cfg.encoder_length : t] for t in selected]
        )
        z = model.encode(
            torch.from_numpy(x_history).to(DEVICE),
            torch.from_numpy(u_history).to(DEVICE),
        )
        Z[selected] = z.cpu().numpy()
    batches.close()
    return Z


def standardize_latents(Z, train_end, cfg):
    valid = np.arange(cfg.encoder_length, train_end)
    mean = np.nanmean(Z[valid], axis=0, keepdims=True)
    std = np.maximum(np.nanstd(Z[valid], axis=0, keepdims=True), 1e-6)
    return ((Z - mean) / std).astype(np.float32), mean, std


def align_latents_to_sensor_time(Z):
    # Z[t]はX[t-1]までを含む。品質入力のindex sでX[s]と揃えるためZ[s+1]を使う。
    aligned = np.full_like(Z, np.nan, dtype=np.float32)
    aligned[:-1] = Z[1:]
    return aligned


def build_quality_features(data, Z_scaled, cfg):
    Z_aligned = align_latents_to_sensor_time(Z_scaled)

    required_start = int(data["feature_times"].min() - cfg.quality_length)
    required_end = int(data["feature_times"].max())
    if not np.isfinite(Z_aligned[required_start:required_end]).all():
        raise ValueError("quality history contains undefined latent values")

    direct = np.concatenate([data["X"], data["U"]], axis=1).astype(np.float32)
    latent = np.concatenate([Z_aligned, data["U"]], axis=1).astype(np.float32)
    return {"direct": direct, "latent": latent}

## 8. 品質推定器

GRUとMLPには同じ64 stepの履歴を渡す。履歴を使うかどうかではなく、推定器の違いを比べる。

- GRU：`batch × 64 × feature_dim` の時系列入力
- MLP：同じ入力を `batch × (64 × feature_dim)` に平坦化

MLPのhidden幅は、対応するGRUと学習パラメータ数が近くなるように自動で選ぶ。単純なMLPでもかなり強そうなので、比較に加える。

In [ ]:
class QualityGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, sequence):
        _, hidden = self.gru(sequence)
        return self.head(hidden[-1]).squeeze(-1)


class QualityMLP(nn.Module):
    def __init__(self, input_dim, sequence_length, hidden_dim):
        super().__init__()
        flattened_dim = input_dim * sequence_length
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, sequence):
        return self.net(sequence).squeeze(-1)


def choose_mlp_hidden_dim(input_dim, sequence_length, target_parameters):
    best_hidden = None
    best_difference = None
    for hidden in range(2, 257):
        candidate = QualityMLP(input_dim, sequence_length, hidden)
        difference = abs(parameter_count(candidate) - target_parameters)
        if best_difference is None or difference < best_difference:
            best_hidden = hidden
            best_difference = difference
    return best_hidden


def make_quality_model(estimator_type, input_dim, cfg):
    if estimator_type == "gru":
        return QualityGRU(input_dim, cfg.hidden_dim)

    if estimator_type == "mlp":
        reference = QualityGRU(input_dim, cfg.hidden_dim)
        target_parameters = parameter_count(reference)
        hidden_dim = choose_mlp_hidden_dim(
            input_dim,
            cfg.quality_length,
            target_parameters,
        )
        return QualityMLP(input_dim, cfg.quality_length, hidden_dim)

    raise ValueError("estimator_type must be 'mlp' or 'gru'")

## 9. 品質推定器の学習と評価

観測品質を教師にして `SmoothL1Loss` で学習する。外れ値の影響を少し抑える狙いがある。

early stoppingもvalidationの観測品質だけで判断する。真の品質は学習とモデル選択に使わず、シミュレーター内の診断指標として保存する。test結果を見てモデルを選ばない。

In [ ]:
@torch.no_grad()
def predict_quality(model, features, data, cfg, split):
    mask = data[f"q_{split}_mask"]
    dataset = QualityDataset(
        features,
        data["q_obs_scaled"],
        data["feature_times"],
        cfg.quality_length,
        mask,
    )
    loader = DataLoader(dataset, batch_size=cfg.quality_batch_size)

    predictions = []
    observed = []
    true_values = []
    times = []

    model.eval()
    for sequence, _, label_indices in loader:
        pred_scaled = model(sequence.to(DEVICE)).cpu().numpy()
        pred = pred_scaled * float(data["q_std"]) + float(data["q_mean"])

        label_indices = label_indices.numpy()
        predictions.extend(pred)
        observed.extend(data["q_obs"][label_indices])
        selected_times = data["feature_times"][label_indices]
        true_values.extend(data["q_true"][selected_times])
        times.extend(selected_times)

    return {
        "prediction": np.asarray(predictions, dtype=np.float32),
        "observed": np.asarray(observed, dtype=np.float32),
        "true": np.asarray(true_values, dtype=np.float32),
        "time": np.asarray(times, dtype=int),
    }


def train_quality_estimator(
    features,
    data,
    cfg,
    seed,
    representation,
    estimator_type,
):
    train_dataset = QualityDataset(
        features,
        data["q_obs_scaled"],
        data["feature_times"],
        cfg.quality_length,
        data["q_train_mask"],
    )
    val_dataset = QualityDataset(
        features,
        data["q_obs_scaled"],
        data["feature_times"],
        cfg.quality_length,
        data["q_val_mask"],
    )
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        raise ValueError("quality dataset is empty")

    set_seed(seed)
    model = make_quality_model(
        estimator_type,
        input_dim=features.shape[1],
        cfg=cfg,
    ).to(DEVICE)

    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.quality_batch_size,
        shuffle=True,
        generator=generator,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.quality_batch_size,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    loss_fn = nn.SmoothL1Loss()

    best_value = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    wait = 0

    description = f"{representation}_{estimator_type} seed={seed}"
    epochs = progress(
        range(cfg.quality_epochs),
        desc=description,
        total=cfg.quality_epochs,
    )

    for epoch in epochs:
        model.train()
        for sequence, target, _ in train_loader:
            sequence = sequence.to(DEVICE)
            target = target.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(sequence), target)
            loss.backward()
            optimizer.step()

            if cfg.run_mode == "smoke":
                break

        model.eval()
        values = []
        with torch.no_grad():
            for sequence, target, _ in val_loader:
                values.append(
                    loss_fn(
                        model(sequence.to(DEVICE)),
                        target.to(DEVICE),
                    ).item()
                )
                if cfg.run_mode == "smoke":
                    break

        value = float(np.mean(values))
        if value < best_value - cfg.min_delta:
            best_value = value
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1

        epochs.set_postfix(val=f"{value:.4f}", best=f"{best_value:.4f}")
        if wait >= cfg.quality_patience:
            break

    epochs.close()
    model.load_state_dict(best_state)
    model.eval()

    val_prediction = predict_quality(model, features, data, cfg, "val")
    test_prediction = predict_quality(model, features, data, cfg, "test")

    return model, best_epoch, val_prediction, test_prediction

## 10. 5 seed比較

各seedでDynamicsを1回だけ学習し、同じlatentを4モデルで共有する。品質推定器のseedと品質ラベル集合も揃える。

モデル以外の違いをなるべく減らして比べる。各seedのMAE、予測値、学習時間はCSVへ保存する。

In [ ]:
raw_rows = []
prediction_rows = []
timing_rows = []

try:
    seed_bar = progress(
        CFG.seeds,
        desc="v14 seeds",
        total=len(CFG.seeds),
        leave=True,
    )

    for process_seed in seed_bar:
        seed_start = time.perf_counter()
        seed_bar.set_postfix(seed=process_seed)

        data = build_data(CFG, process_seed)

        dynamics_start = time.perf_counter()
        dynamics, dyn_val_mse, persistence_val_mse, dyn_best_epoch = train_dynamics(
            data,
            CFG,
            process_seed,
        )
        Z = compute_latents(dynamics, data["X"], data["U"], CFG)
        Z_scaled, _, _ = standardize_latents(Z, data["train_end"], CFG)
        feature_sets = build_quality_features(data, Z_scaled, CFG)
        dynamics_seconds = time.perf_counter() - dynamics_start

        quality_seed = process_seed + 100_000
        model_specs = [
            ("direct", "mlp"),
            ("direct", "gru"),
            ("latent", "mlp"),
            ("latent", "gru"),
        ]

        for representation, estimator_type in model_specs:
            model_name = f"{representation}_{estimator_type}"
            model_start = time.perf_counter()

            model, best_epoch, val_pred, test_pred = train_quality_estimator(
                feature_sets[representation],
                data,
                CFG,
                quality_seed,
                representation,
                estimator_type,
            )
            elapsed = time.perf_counter() - model_start

            raw_rows.append({
                "process_seed": process_seed,
                "quality_seed": quality_seed,
                "model": model_name,
                "representation": representation,
                "quality_estimator": estimator_type,
                "parameter_count": parameter_count(model),
                "quality_best_epoch": best_epoch,
                "n_train_labels": data["n_train_labels"],
                "n_val_labels": data["n_val_labels"],
                "n_test_labels": data["n_test_labels"],
                "val_obs_mae": safe_mae(val_pred["observed"], val_pred["prediction"]),
                "val_true_mae_diagnostic": safe_mae(val_pred["true"], val_pred["prediction"]),
                "test_obs_mae": safe_mae(test_pred["observed"], test_pred["prediction"]),
                "test_true_mae_diagnostic": safe_mae(test_pred["true"], test_pred["prediction"]),
                "dyn_val_mse": dyn_val_mse,
                "persistence_val_mse": persistence_val_mse,
                "dyn_best_epoch": dyn_best_epoch,
                "quality_training_seconds": elapsed,
            })

            for t, pred, obs, true in zip(
                test_pred["time"],
                test_pred["prediction"],
                test_pred["observed"],
                test_pred["true"],
            ):
                prediction_rows.append({
                    "process_seed": process_seed,
                    "model": model_name,
                    "feature_time": int(t),
                    "prediction": float(pred),
                    "observed_quality": float(obs),
                    "true_quality_diagnostic": float(true),
                })

        timing_rows.append({
            "process_seed": process_seed,
            "dynamics_and_latent_seconds": dynamics_seconds,
            "total_seed_seconds": time.perf_counter() - seed_start,
        })

    seed_bar.close()

    raw_results = pd.DataFrame(raw_rows)
    predictions = pd.DataFrame(prediction_rows)
    timings = pd.DataFrame(timing_rows)

    save_csv("raw_results.csv", raw_results)
    save_csv("test_predictions.csv", predictions)
    save_csv("timings.csv", timings)

except Exception:
    (RUN_DIR / "FAILED").write_text(traceback.format_exc(), encoding="utf-8")
    raise

## 11. 結果の集計

モデルごとの平均、標準偏差、seed数を `summary.csv` にまとめる。

`paired_comparisons.csv` には、同じseed内でのMAE差を保存する。定義は `left - right` なので、負ならleft側が良い。平均値だけでなく、seedごとの勝ち負けも確認できる。

In [ ]:
summary = (
    raw_results
    .groupby(["model", "representation", "quality_estimator"])
    .agg(
        test_true_mae_mean=("test_true_mae_diagnostic", "mean"),
        test_true_mae_std=("test_true_mae_diagnostic", "std"),
        test_obs_mae_mean=("test_obs_mae", "mean"),
        test_obs_mae_std=("test_obs_mae", "std"),
        val_obs_mae_mean=("val_obs_mae", "mean"),
        parameter_count_mean=("parameter_count", "mean"),
        seed_count=("process_seed", "count"),
    )
    .reset_index()
)

comparison_pairs = [
    ("latent_mlp", "direct_mlp", "representation_effect_mlp"),
    ("latent_gru", "direct_gru", "representation_effect_gru"),
    ("direct_gru", "direct_mlp", "estimator_effect_direct"),
    ("latent_gru", "latent_mlp", "estimator_effect_latent"),
]

paired_rows = []
pivot = raw_results.pivot(
    index="process_seed",
    columns="model",
    values="test_true_mae_diagnostic",
)

for left, right, comparison in comparison_pairs:
    delta = pivot[left] - pivot[right]
    paired_rows.append({
        "comparison": comparison,
        "left_model": left,
        "right_model": right,
        "definition": "left MAE - right MAE; negative favors left",
        "mean_delta": float(delta.mean()),
        "std_delta": float(delta.std(ddof=1)) if len(delta) > 1 else np.nan,
        "left_wins": int((delta < 0).sum()),
        "ties": int((delta == 0).sum()),
        "seed_count": int(delta.notna().sum()),
    })

paired = pd.DataFrame(paired_rows)

save_csv("summary.csv", summary)
save_csv("paired_comparisons.csv", paired)

display(raw_results)
display(summary)
display(paired)

## 12. 図と終了記録

4モデルの真値MAEを棒グラフで保存する。5 seedで比べた結果では、標本標準偏差を誤差線に使う。

学習データの使い方は `audit.json` に残す。正常終了なら `COMPLETE`、途中で失敗したら `FAILED` を保存する。

In [ ]:
plot_order = ["direct_mlp", "direct_gru", "latent_mlp", "latent_gru"]
plot_data = summary.set_index("model").loc[plot_order]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(
    plot_order,
    plot_data["test_true_mae_mean"],
    yerr=plot_data["test_true_mae_std"].fillna(0.0),
    capsize=4,
)
ax.set_ylabel("Test true-quality MAE (diagnostic)")
ax.set_title("Quality estimator comparison")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(RUN_DIR / "model_comparison.png", dpi=160)
plt.show()

audit = {
    "old_simulator_used": True,
    "fixed_condition": {
        "drift_mode": "AR",
        "disturbance_target": "output",
        "disturbance_regime": "persistent",
        "z_dim": CFG.z_dim,
        "train_label_count": CFG.train_label_count,
    },
    "q_true_used_for_training": False,
    "true_state_used_for_training": False,
    "true_drift_used_for_training": False,
    "model_selection_metric": "validation observed-quality SmoothL1 loss",
    "test_used_for_model_selection": False,
    "quality_split": "train by arrival time; validation/test by feature-time window",
    "same_quality_labels_for_all_models": True,
    "same_quality_seed_for_all_models": True,
    "same_history_length_for_mlp_and_gru": True,
    "mlp_history_handling": "flattened fixed-length sequence",
    "mlp_parameter_matching": "hidden width selected to approximate corresponding GRU parameter count",
    "latent_model_frozen_during_quality_training": True,
}
save_json("audit.json", audit)

(RUN_DIR / "RUNNING").unlink(missing_ok=True)
(RUN_DIR / "COMPLETE").write_text("complete\n", encoding="utf-8")

print("saved to:", RUN_DIR.resolve())
print("complete:", (RUN_DIR / "COMPLETE").exists())

## 13. 結果の見方

結果を見るときは、まず `paired_comparisons.csv` を確認する。

- `representation_effect_mlp`：MLPでの `Latent - Direct`
- `representation_effect_gru`：GRUでの `Latent - Direct`
- `estimator_effect_direct`：生センサ入力での `GRU - MLP`
- `estimator_effect_latent`：latent入力での `GRU - MLP`

差が負なら左側のモデルが良い。真値MAEはシミュレーター内だけで見られるので、観測品質MAEも一緒に確認する。

今の結果ではDirect MLPがかなり強いみたい。Latent GRUとの平均差は小さく、seedごとに勝ち負けが入れ替わる。潜在表現がいつも有利とはまだ言えない。